# Clasificación de Sentimientos de Reseñas de Películas en IMDb con Redes Neuronales Recurrentes (PyTorch)
**Maestría en Inteligencia Artificial – Técnicas de Deep Learning**

Los ratings y reseñas de películas son una fuente rica de datos textuales que pueden analizarse para identificar el sentimiento subyacente (positivo o negativo). Las redes neuronales recurrentes (RNN) son eficaces para clasificar secuencias de texto, pues capturan información contextual en oraciones y palabras e identifican patrones relevantes para determinar la polaridad de una reseña.

## Redes Neuronales Recurrentes
Las RNN son adecuadas para datos secuenciales como el texto, ya que almacenan y recuerdan información previa de la secuencia, lo cual es crucial para entender el contexto. Aplicadas a reseñas de películas, pueden entrenarse para identificar palabras, frases y patrones asociados a sentimientos positivos o negativos.

## Procesamiento de Texto
Se convierten las reseñas a un formato numérico mediante tokenización y vectorización de palabras (embeddings). Luego la RNN analiza la secuencia de palabras para clasificar cada reseña en positiva o negativa.

## Ventajas y Aplicaciones
- **Detección precisa de emociones:** captura sentimientos más matizados en textos largos y complejos.
- **Adaptabilidad a distintos idiomas y contextos:** pueden entrenarse con diferentes conjuntos de datos (géneros de películas, plataformas de reseñas).
- **Asistencia en sistemas de recomendación:** mejora las recomendaciones personalizadas en plataformas de streaming.

## A. Objetivo
Desarrollar un método basado en redes neuronales recurrentes que permita clasificar con precisión las reseñas de películas en IMDb en **positiva** o **negativa**.

## B. Conjunto de datos
Dataset de reseñas de IMDb (Large Movie Review Dataset, Stanford `aclImdb`), con 50.000 reseñas etiquetadas con su sentimiento. Se descarga automáticamente en la sección 1.

## C. Actividades por realizar
1. Preprocesamiento de las reseñas (limpieza, tokenización, secuencias numéricas y embeddings Word2Vec/GloVe).
2. Desarrollo de la arquitectura RNN (LSTM o GRU) justificando cada capa.
3. Entrenamiento y evaluación (train/val/test; precisión, recall y F1-score).

In [ ]:
# Instalación de dependencias (descomentar si hace falta, p. ej. en Colab)
# !pip install -q torch gensim scikit-learn matplotlib seaborn pandas

In [ ]:
import os, re, random, tarfile, copy, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from gensim.models import Word2Vec

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| dispositivo:", device)

## 1. Carga del dataset
Se descarga el dataset `aclImdb` y se leen las 50.000 reseñas (25.000 de `train` y 25.000 de `test` del original); después se re-particionarán en entrenamiento / validación / prueba propios.

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
tar_path = os.path.join(data_dir, "aclImdb_v1.tar.gz")
base_dir = os.path.join(data_dir, "aclImdb")

if not os.path.isdir(base_dir):
    if not os.path.exists(tar_path):
        urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)

def load_split(split):
    texts, labels = [], []
    for label_name, label in [("pos", 1), ("neg", 0)]:
        folder = os.path.join(base_dir, split, label_name)
        for fname in os.listdir(folder):
            with open(os.path.join(folder, fname), encoding="utf-8") as f:
                texts.append(f.read())
            labels.append(label)
    return texts, labels

t1, l1 = load_split("train")
t2, l2 = load_split("test")
df = pd.DataFrame({"review": t1 + t2, "sentiment": l1 + l2}).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
print(df["sentiment"].value_counts())
df["n_words"] = df["review"].str.split().str.len()
print(df["n_words"].describe())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x="sentiment", data=df, ax=ax[0]); ax[0].set_title("Distribución de clases (0=neg, 1=pos)")
sns.histplot(df["n_words"], bins=60, ax=ax[1]); ax[1].set_title("Longitud de reseñas (palabras)")
plt.tight_layout(); plt.show()

## 2. Preprocesamiento de las reseñas (Actividad 1)
Pipeline:
1. **Limpieza:** minúsculas, eliminación de etiquetas HTML (`<br />`), URLs y caracteres no alfabéticos (se conservan apóstrofes y signos `!` `?` por su carga emocional).
2. **Tokenización:** separación en palabras.
3. **Secuencias numéricas:** vocabulario construido a mano (índice 0 = `<PAD>`, 1 = `<UNK>`), con *padding*/*truncating* a longitud fija.
4. **Embeddings:** Word2Vec entrenado sobre el corpus de entrenamiento (alternativa: GloVe preentrenado).

Primero se hace la partición para evitar *data leakage*: el vocabulario y Word2Vec se ajustan **solo con el conjunto de entrenamiento**.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)            # HTML
    text = re.sub(r"http\S+|www\S+", " ", text)     # URLs
    text = re.sub(r"[^a-z!?' ]+", " ", text)         # solo letras y signos útiles
    text = re.sub(r"([!?])", r" \1 ", text)          # separar ! y ?
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean"] = df["review"].apply(clean_text)
df["tokens"] = df["clean"].str.split()
print(df.loc[0, "review"][:300])
print(df.loc[0, "tokens"][:30])

In [ ]:
# Partición: 70% train / 15% val / 15% test (estratificada)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["sentiment"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["sentiment"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
MAX_WORDS = 20000   # tamaño del vocabulario (incluye <PAD> y <UNK>)
MAX_LEN = 300       # longitud de secuencia
EMB_DIM = 100
PAD, UNK = 0, 1

# Vocabulario construido solo con el conjunto de entrenamiento
counter = Counter(tok for toks in train_df["tokens"] for tok in toks)
most_common = [w for w, _ in counter.most_common(MAX_WORDS - 2)]
word2idx = {w: i + 2 for i, w in enumerate(most_common)}
print("Tamaño del vocabulario:", len(word2idx) + 2)

def encode(tokens):
    ids = [word2idx.get(t, UNK) for t in tokens][:MAX_LEN]     # truncating post
    return ids + [PAD] * (MAX_LEN - len(ids))                   # padding post

def to_tensor(frame):
    X = torch.tensor([encode(t) for t in frame["tokens"]], dtype=torch.long)
    y = torch.tensor(frame["sentiment"].values, dtype=torch.float32)
    return X, y

X_train, y_train = to_tensor(train_df)
X_val, y_val = to_tensor(val_df)
X_test, y_test = to_tensor(test_df)
print(X_train.shape, X_val.shape, X_test.shape)

BATCH_SIZE = 128
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

In [ ]:
# Word2Vec entrenado con el corpus de entrenamiento
w2v = Word2Vec(sentences=train_df["tokens"].tolist(), vector_size=EMB_DIM,
               window=5, min_count=2, workers=4, sg=1, epochs=10, seed=SEED)
print("Vocabulario Word2Vec:", len(w2v.wv))
print(w2v.wv.most_similar("great", topn=5))
print(w2v.wv.most_similar("terrible", topn=5))

# Matriz de embeddings alineada con los índices del vocabulario
embedding_matrix = np.zeros((MAX_WORDS, EMB_DIM), dtype=np.float32)
hits = 0
for word, idx in word2idx.items():
    if word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]
        hits += 1
embedding_matrix[UNK] = np.random.normal(scale=0.1, size=EMB_DIM)   # <UNK> aleatorio; <PAD> queda en ceros
print(f"Palabras con vector: {hits}/{len(word2idx)}")

## 3. Arquitectura de la red neuronal recurrente (Actividad 2)
| Capa (PyTorch) | Justificación |
|---|---|
| `nn.Embedding` (inicializada con Word2Vec, `padding_idx=0`) | Convierte índices en vectores densos que capturan similitud semántica; inicializar con Word2Vec acelera la convergencia. Se deja entrenable (*fine-tuning*) para adaptarse a la tarea. |
| `nn.Dropout` (sobre embeddings) | Regulariza la entrada a la red recurrente y reduce el sobreajuste. |
| `nn.LSTM(bidirectional=True)` + `pack_padded_sequence` | La LSTM mitiga el desvanecimiento del gradiente y retiene dependencias largas; la versión bidireccional lee la reseña en ambos sentidos, útil porque el sentimiento depende del contexto previo y posterior (p. ej. negaciones). El *packing* hace que el *padding* no contamine el estado final. |
| `nn.Dropout` | Regularización adicional sobre la representación de la secuencia. |
| `nn.Linear(256, 64)` + `ReLU` | Combina no linealmente las características extraídas por la LSTM. |
| `nn.Linear(64, 1)` + sigmoide | Salida probabilística para clasificación binaria. Por estabilidad numérica la red devuelve *logits* y se entrena con `BCEWithLogitsLoss` (que aplica la sigmoide internamente); en inferencia se aplica `torch.sigmoid`. |

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128, dense_size=64, emb_dropout=0.3, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False, padding_idx=PAD)
        self.emb_dropout = nn.Dropout(emb_dropout)
        self.lstm = nn.LSTM(embedding_matrix.shape[1], hidden_size, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size * 2, dense_size)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.out = nn.Linear(dense_size, 1)

    def forward(self, x):
        lengths = (x != PAD).sum(dim=1).clamp(min=1).cpu()
        emb = self.emb_dropout(self.embedding(x))
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)          # último estado de ambas direcciones
        h = self.dropout(h)
        h = self.dropout2(self.relu(self.fc1(h)))
        return self.out(h).squeeze(1)                      # logits

model = SentimentRNN(embedding_matrix).to(device)
print(model)
print("Parámetros entrenables:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Entrenamiento y evaluación (Actividad 3)
Se usa `Adam`, *gradient clipping*, `ReduceLROnPlateau` y *early stopping* sobre la pérdida de validación, conservando los pesos del mejor modelo.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

def run_epoch(loader, train=False):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += ((logits > 0).float() == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n

EPOCHS, PATIENCE = 10, 3
history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
best_val, best_state, wait = float("inf"), None, 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step(va_loss)
    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)
    print(f"Época {epoch:02d} | loss {tr_loss:.4f} acc {tr_acc:.4f} | val_loss {va_loss:.4f} val_acc {va_acc:.4f}")

    if va_loss < best_val:
        best_val, best_state, wait = va_loss, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, m in enumerate(["loss", "accuracy"]):
    ax[i].plot(history[m], label="train")
    ax[i].plot(history["val_" + m], label="val")
    ax[i].set_title(m); ax[i].set_xlabel("época"); ax[i].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Evaluación sobre el conjunto de prueba
model.eval()
probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        probs.append(torch.sigmoid(model(xb.to(device))).cpu())
y_prob = torch.cat(probs).numpy()
y_true = y_test.numpy().astype(int)
y_pred = (y_prob >= 0.5).astype(int)

print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
print(f"F1-score : {f1_score(y_true, y_pred):.4f}\n")
print(classification_report(y_true, y_pred, target_names=["negativa", "positiva"], digits=4))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["negativa", "positiva"], yticklabels=["negativa", "positiva"])
plt.xlabel("Predicho"); plt.ylabel("Real"); plt.title("Matriz de confusión (test)"); plt.show()

In [ ]:
# Predicciones sobre reseñas nuevas
def predict_sentiment(text):
    ids = torch.tensor([encode(clean_text(text).split())], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(ids)).item()
    return ("positiva" if p >= 0.5 else "negativa"), round(p, 3)

for r in ["An absolute masterpiece, the acting was brilliant and I loved every minute!",
          "Boring, predictable and way too long. A complete waste of time.",
          "Not bad, but I expected much more from the director."]:
    print(predict_sentiment(r), "<-", r)

## 5. Conclusiones
- Se construyó un pipeline completo en PyTorch: limpieza, tokenización, vocabulario, secuencias con *padding* y embeddings Word2Vec entrenados solo con el conjunto de entrenamiento.
- Una **BiLSTM** con embeddings preentrenados clasifica las reseñas con las métricas de accuracy, precisión, recall y F1 reportadas arriba (usualmente ~87–90 % en este dataset).
- **Posibles mejoras:** embeddings GloVe preentrenados, mecanismos de atención, mayor `MAX_LEN`, ajuste de hiperparámetros o modelos basados en Transformers.